In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import yaml

epoch_better_path = "../../experiment_data/balance_metrics/tvcg/multiseed/loss_1e6.csv"

epoch_data = pd.read_csv(epoch_better_path)

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt

import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "ggplot2"

color_seq = px.colors.qualitative.Dark24
color_seq_train = px.colors.qualitative.Pastel
color_seq_val = px.colors.qualitative.Set1
epoch_data.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,dataset,split,model,k,epoch,layer,thresh,thresh_mode,node_count,average_branching_factor,...,average_colless_index,total_colless,missing_colless,missing_colless_frac,sackin_index,average_sackin_index,leaf_count,total_volume,train_acc,val_acc
0,mnist,val,resnet_c,20,40,a4,0.000001,0.000001,264,2.023077,...,462.320000,125,5,0.038462,1011941.0,7551.798507,134,10000,0.991083,0.9751
1,mnist,val,resnet_c,20,104,a5,0.000001,0.000001,166,2.062500,...,1404.452055,73,7,0.087500,644397.0,7492.988372,86,10000,0.999567,0.9787
2,mnist,train,resnet_c,20,48,a5,0.000001,0.000001,1490,2.062327,...,3109.985185,675,47,0.065097,36369878.0,47356.611979,768,60000,0.994117,0.9759
3,mnist,train,resnet_c,20,32,a5,0.000001,0.000001,1525,2.067843,...,2180.119534,686,51,0.069199,36533236.0,46361.974619,788,60000,0.985617,0.9740
4,mnist,trainUval,resnet_c,20,128,a5,0.000001,0.000001,803,2.040712,...,11319.981333,375,18,0.045802,21712056.0,52956.234146,410,70000,0.999667,0.9804


In [30]:
import plotly.graph_objects as go

epoch_trainUval = epoch_data[(epoch_data["split"] == "trainUval") & (epoch_data["layer"] == "a1")]
epoch_trainUval = epoch_trainUval.sort_values(by='epoch')

fig1 = make_subplots(specs=[[{"secondary_y": True}]], )

for i, (ds, df) in enumerate(epoch_trainUval.groupby("model")):
    run = ds.split("_")[-1].capitalize()
    # train_acc = go.Scatter(x=df["epoch"], y=df["train_acc"], name=f"Train Acc {run}", line=dict(color=color_seq_train[i], width=3), legendgroup=run)
    val_acc = go.Scatter(x=df["epoch"], y=df["val_acc"], name=f"Accuracy {run}", line=dict(color=color_seq_val[i], width=3), legendgroup=run, opacity=0.7)
    sackin = go.Scatter(x=df["epoch"], y=df["sackin_index"], name=f"Sackin {run}", mode='markers', marker=dict(color=color_seq[i], size=8, opacity=0.6), legendgroup=run)

    fig1.add_trace(sackin, secondary_y=False)
    # fig1.add_trace(train_acc, secondary_y=True)
    fig1.add_trace(val_acc, secondary_y=True)

fig1.update_annotations(font_size=16)
fig1.update_layout(margin=dict(l=0, r=0, t=0, b=0), width=420*2, height=180*2, font=dict(size=22), showlegend=True, legend=dict(
    xanchor="right", yanchor="top", x=1.4, y=0.98, bgcolor="rgba(255,255,255,0.5)", font=dict(size=15), title=""))
fig1.update_xaxes(title_text="Epoch", title_standoff=18, automargin=True, range=[0,150])
fig1.update_yaxes(title_text="Sackin Index", type="log", title_standoff=18, automargin=True, secondary_y=False)
fig1.update_yaxes(title_text="Accuracy", title_standoff=18, automargin=True, secondary_y=True, nticks=10, range=[0, 1.05])
fig1.show()
fig1.write_image(f"multiseed_epochs.png", scale=4)
